# ⚡ RAIZEN: Enterprise Full-Stack Coding Intelligence
### 🚀 High-Performance GPU Streaming Backend & Cloudflare Tunnel Engine

---

<p align="center">
  <b>Architected, Fine-Tuned & Created by <a href="https://shawaz.vercel.app/" target="_blank">SHAWAZ</a></b>
</p>

<p align="center">
  <a href="https://shawaz.vercel.app/" target="_blank"><img src="https://img.shields.io/badge/Creator-SHAWAZ-blue.svg?style=for-the-badge" alt="Creator"></a>
  <a href="https://shawaz.vercel.app/" target="_blank"><img src="https://img.shields.io/badge/Portfolio-shawaz.vercel.app-green.svg?style=for-the-badge" alt="Portfolio"></a>
  <a href="https://huggingface.co/shawaz03/RAIZEN" target="_blank"><img src="https://img.shields.io/badge/HuggingFace-shawaz03%2FRAIZEN-orange.svg?style=for-the-badge" alt="HuggingFace"></a>
  <img src="https://img.shields.io/badge/Model-RAIZEN--7B-red.svg?style=for-the-badge" alt="Model">
  <img src="https://img.shields.io/badge/Precision-4--bit_NF4-purple.svg?style=for-the-badge" alt="Precision">
  <img src="https://img.shields.io/badge/Hosting_Cost-%240.00%20(Free%20GPU)-gold.svg?style=for-the-badge" alt="Cost">
</p>

---

## 📖 Instructions
1. **Enable Free GPU**: Go to `Runtime` ➔ `Change runtime type` ➔ Select **T4 GPU** (or A100 if available).
2. **Start Backend Engine**: Click `Runtime` ➔ `Run all` (or press `Ctrl + F9`).
3. **Connect Frontend**: Copy the generated **Cloudflare Tunnel URL** (`https://*.trycloudflare.com`) and paste it into the **RAIZEN Chat Studio** frontend.

---

In [ ]:
# Cell 1: Environment & Dependency Installation
print("📦 [1/4] Installing RAIZEN core dependencies (FastAPI, Transformers, BitsAndBytes, Accelerate)...")
!pip install -q -U transformers accelerate bitsandbytes peft fastapi uvicorn pydantic
print("✅ Core dependencies installed successfully!")

In [ ]:
# Cell 2: Download & Configure Cloudflare Quick Tunnel
print("🌐 [2/4] Downloading & configuring Cloudflare Quick Tunnel binary...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version
print("✅ Cloudflare Tunnel binary is configured and ready for zero-config public tunneling!")

In [ ]:
# Cell 3: Load RAIZEN Tokenizer
import os
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TextIteratorStreamer

MODEL_ID = "shawaz03/RAIZEN"

print(f"📥 [3/4] Initializing RAIZEN Tokenizer from Hugging Face ({MODEL_ID})...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer ready! Vocab size: {len(tokenizer):,}, Pad token: '{tokenizer.pad_token}'")

In [ ]:
# Cell 4: Load RAIZEN 7B in 4-bit NF4 Quantization
print(f"🧠 [4/4] Loading RAIZEN 7B in 4-bit NF4 precision onto GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
print(f"🎉 RAIZEN 7B loaded successfully on {gpu_name}!")
print(f"📊 GPU VRAM Allocated: {vram_gb:.2f} GB (Leaves ~10 GB free for 4K context on free Colab T4)")

In [ ]:
# Cell 5: Warmup Inference & CUDA Kernel Pre-Compilation
import time

print("⚡ Pre-compiling CUDA kernels with warmup inference...")
warmup_messages = [
    {"role": "system", "content": "You are RAIZEN, an elite AI coding intelligence created by SHAWAZ (https://shawaz.vercel.app/)."},
    {"role": "user", "content": "Hello RAIZEN, confirm your operational readiness in one sentence."}
]

t0 = time.time()
warmup_text = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
warmup_inputs = tokenizer(warmup_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    warmup_out = model.generate(
        **warmup_inputs,
        max_new_tokens=32,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id
    )

warmup_resp = tokenizer.decode(warmup_out[0][warmup_inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
elapsed = time.time() - t0

print(f"✅ Warmup successful in {elapsed:.2f}s!")
print(f"🤖 RAIZEN Warmup Output: \"{warmup_resp}\"")

In [ ]:
# Cell 6: FastAPI Application & Cross-Origin Resource Sharing (CORS) Setup
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import threading
import json

app = FastAPI(
    title="RAIZEN Coding Intelligence API",
    description="High-performance streaming inference engine architected by SHAWAZ (https://shawaz.vercel.app/)",
    version="1.0.0"
)

# Enable open CORS for Next.js frontend connection
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("✅ FastAPI application initialized with universal CORS middleware!")

In [ ]:
# Cell 7: Health Check & System Status Endpoints
import time

@app.get("/")
@app.get("/health")
async def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "healthy",
        "model": "RAIZEN-7B",
        "version": "1.0.0",
        "creator": "SHAWAZ",
        "portfolio": "https://shawaz.vercel.app/",
        "huggingface": "https://huggingface.co/shawaz03/RAIZEN",
        "gpu": gpu_name,
        "vram_allocated_gb": round(vram_gb, 2),
        "precision": "4-bit NF4 (bitsandbytes)",
        "timestamp": int(time.time()),
    }

print("✅ Health check endpoints (/ and /health) configured!")